# forge-kernels quickstart: patch a Qwen model and measure the delta

This notebook loads `Qwen/Qwen2.5-0.5B`, runs a few training steps to get a
baseline, then applies `forge.patch` and runs the same steps again. It prints the
speedup and peak-memory delta so you can see the effect live.

**Requirements:** a CUDA GPU (results in the README are from an A100-80GB), plus
`forge-kernels` and `transformers`. Install from the repo root:

```bash
uv sync --dev
# or: pip install -e . transformers
```

The headline result on an A100-80GB is roughly **1.20x faster training and 36%
less memory** at batch 4 x seq 1024, driven by fused linear cross-entropy. Your
numbers depend on your GPU, batch size, and sequence length.


In [ ]:
import torch

assert torch.cuda.is_available(), "This notebook needs a CUDA GPU."
device = "cuda"
print("torch:", torch.__version__)
print("device:", torch.cuda.get_device_name(0))

MODEL_ID = "Qwen/Qwen2.5-0.5B"
BATCH, SEQ = 4, 1024   # lower these if you hit an out-of-memory error
WARMUP, STEPS = 3, 10


## Load the model and build one training batch

We use a fixed random batch with `labels = input_ids` so every step exercises the
full forward, cross-entropy loss, and backward. The batch is identical for the
baseline and patched runs, so the comparison is apples to apples.


In [ ]:
from transformers import AutoModelForCausalLM, AutoConfig

config = AutoConfig.from_pretrained(MODEL_ID)
vocab = config.vocab_size

def load_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16)
    return model.to(device).train()

g = torch.Generator(device=device).manual_seed(0)
input_ids = torch.randint(0, vocab, (BATCH, SEQ), generator=g, device=device)
batch = {
    "input_ids": input_ids,
    "attention_mask": torch.ones_like(input_ids),
    "labels": input_ids.clone(),
}
print(f"batch: {BATCH} x {SEQ} tokens, vocab={vocab}")


## Measurement helper

`measure` runs a few warmup steps, then times `STEPS` full training steps with
CUDA events and records peak GPU memory. It returns milliseconds per step and
peak megabytes. Each step is forward, backward, optimizer step, and zero-grad.


In [ ]:
def measure(model, warmup=WARMUP, steps=STEPS):
    opt = torch.optim.AdamW(model.parameters(), lr=1e-5)

    def train_step():
        opt.zero_grad(set_to_none=True)
        loss = model(**batch).loss
        loss.backward()
        opt.step()
        return loss

    for _ in range(warmup):
        train_step()
    torch.cuda.synchronize()

    torch.cuda.reset_peak_memory_stats()
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(steps):
        last_loss = train_step()
    end.record()
    torch.cuda.synchronize()

    ms_per_step = start.elapsed_time(end) / steps
    peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    return ms_per_step, peak_mb, float(last_loss)


## Baseline (unpatched, eager PyTorch)


In [ ]:
model = load_model()
base_ms, base_mb, base_loss = measure(model)
print(f"eager:   {base_ms:7.2f} ms/step   {base_mb:8.1f} MB peak   loss={base_loss:.4f}")

del model
torch.cuda.empty_cache()


## Patched with forge

`forge.patch` swaps the kernels into the model in place. We pass `min_elements=0`
so every shape-sensitive kernel is patched unconditionally, which is what you want
when benchmarking. In normal use, drop that argument and forge falls back to eager
on calls too small for fusion to pay off. `collect_stats=True` records how many
calls hit the fused path.


In [ ]:
import forge

model = load_model()
forge.patch(model, min_elements=0, collect_stats=True)

fused_ms, fused_mb, fused_loss = measure(model)
print(f"forge:   {fused_ms:7.2f} ms/step   {fused_mb:8.1f} MB peak   loss={fused_loss:.4f}")
print("patched counts:", getattr(model, "_forge_patched_counts", {}))


## Result


In [ ]:
speedup = base_ms / fused_ms
mem_saved = (1 - fused_mb / base_mb) * 100

print(f"{'':10} {'ms/step':>10} {'peak MB':>10}")
print(f"{'eager':10} {base_ms:>10.2f} {base_mb:>10.1f}")
print(f"{'forge':10} {fused_ms:>10.2f} {fused_mb:>10.1f}")
print("-" * 32)
print(f"speedup: {speedup:.2f}x")
print(f"memory saved: {mem_saved:.1f}%")

# Loss tracks the baseline closely. Small drift is expected here because it
# accumulates over optimizer steps; per-step forward parity is checked in the
# kernel test suites, not in this timing loop.
print(f"loss delta: {abs(fused_loss - base_loss):.2e}")


## Notes

- Call `forge.unpatch(model)` to restore the original forwards at any time.
- The batch uses random tokens, so the loss value itself is not meaningful. This
  notebook measures speed and memory, not convergence. For a real fine-tuning run
  see the demos under `forge/demos/`.
- Most of the memory saving comes from fused linear cross-entropy, which never
  materializes the full `(batch x seq, vocab)` logits tensor. The effect grows
  with vocabulary size and sequence length. See
  [`../docs/benchmarks/benchmarks.md`](../docs/benchmarks/benchmarks.md).
- Speedup and memory depend on your GPU, batch, and sequence length. Small shapes
  can show little or no speedup, which is expected and documented in
  [`../docs/api-reference/patching.md`](../docs/api-reference/patching.md).
